# DC-DC Converters (Choppers)

This chapter simulates ideal Buck, Boost, and Buck-Boost converters in both CCM and DCM.

Topics covered:
- Duty ratio switching functions
- Inductor current ripple and capacitor ripple
- Comparison of CCM and DCM transfer characteristics

## Core Theory

Define conversion ratio $M = V_o / V_g$ and duty cycle $D$.

1. Buck:
- CCM: $M=D$
- DCM: $M=\dfrac{2}{1+\sqrt{1+4K/D^2}}$

2. Boost:
- CCM: $M=\dfrac{1}{1-D}$
- DCM: $M=\dfrac{1}{2}(1+\sqrt{1+4D^2/K})$

3. Inverting Buck-Boost:
- CCM: $M=-\dfrac{D}{1-D}$
- DCM: $M=-\dfrac{D}{\sqrt{K}}$

where $K=\dfrac{2Lf_s}{R}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from power_sim import dc_dc, plotter

In [ ]:
# Common parameters
Vg = 48.0
D = 0.45
fs = 40_000.0
R = 12.0
L = 180e-6
C = 220e-6

# Simulate several switching cycles
Tsw = 1.0 / fs
t = np.linspace(0.0, 8.0 * Tsw, 5000, endpoint=False)

## Buck Converter: CCM vs DCM

In CCM, inductor current does not reach zero; in DCM it does.

In [ ]:
buck_ccm = dc_dc.buck_converter(t, Vg, D, fs, R, L, capacitance=C, mode='CCM')
buck_dcm = dc_dc.buck_converter(t, Vg, D, fs, R, L, capacitance=C, mode='DCM')

print('Buck CCM theory:', buck_ccm['theory'])
print('Buck DCM theory:', buck_dcm['theory'])
print('Buck CCM metrics:', buck_ccm['metrics'])

In [ ]:
fig, _ = plotter.plot_stacked_waveforms(
    x=t * 1e6,
    traces=[
        {'y': buck_ccm['switch'], 'label': 'Switch command', 'ylabel': 's(t)'},
        {'y': buck_ccm['inductor_current'], 'label': 'i_L CCM', 'ylabel': 'i_L (A)'},
        {'y': buck_dcm['inductor_current'], 'label': 'i_L DCM', 'ylabel': 'i_L (A)'},
        {'y': buck_ccm['output_voltage'], 'label': 'v_o CCM', 'ylabel': 'v_o (V)'},
    ],
    xlabel='Time (us)',
    title='Buck Converter Waveforms'
)
plt.show()

## Boost Converter

When the switch is ON, inductor stores energy. When OFF, inductor adds to source through diode to produce $V_o>V_g$.

In [ ]:
boost_ccm = dc_dc.boost_converter(t, Vg, D, fs, R, L, capacitance=C, mode='CCM')
boost_dcm = dc_dc.boost_converter(t, Vg, D, fs, R, L, capacitance=C, mode='DCM')

print('Boost CCM theory:', boost_ccm['theory'])
print('Boost DCM theory:', boost_dcm['theory'])
print('Boost CCM metrics:', boost_ccm['metrics'])

In [ ]:
fig, _ = plotter.plot_stacked_waveforms(
    x=t * 1e6,
    traces=[
        {'y': boost_ccm['switch_current'], 'label': 'Switch current', 'ylabel': 'i_sw (A)'},
        {'y': boost_ccm['diode_current'], 'label': 'Diode current', 'ylabel': 'i_D (A)'},
        {'y': boost_ccm['inductor_voltage'], 'label': 'Inductor voltage', 'ylabel': 'v_L (V)'},
        {'y': boost_ccm['output_voltage'], 'label': 'Output voltage', 'ylabel': 'v_o (V)'},
    ],
    xlabel='Time (us)',
    title='Boost Converter CCM Waveforms'
)
plt.show()

## Inverting Buck-Boost Converter

Output voltage polarity is negative with respect to source reference.

In [ ]:
bb_ccm = dc_dc.buck_boost_converter(t, Vg, D, fs, R, L, capacitance=C, mode='CCM')
bb_dcm = dc_dc.buck_boost_converter(t, Vg, D, fs, R, L, capacitance=C, mode='DCM')

print('Buck-Boost CCM theory:', bb_ccm['theory'])
print('Buck-Boost DCM theory:', bb_dcm['theory'])
print('Buck-Boost CCM metrics:', bb_ccm['metrics'])

In [ ]:
fig, _ = plotter.plot_stacked_waveforms(
    x=t * 1e6,
    traces=[
        {'y': bb_ccm['inductor_current'], 'label': 'i_L CCM', 'ylabel': 'i_L (A)'},
        {'y': bb_dcm['inductor_current'], 'label': 'i_L DCM', 'ylabel': 'i_L (A)'},
        {'y': bb_ccm['output_voltage'], 'label': 'v_o CCM', 'ylabel': 'v_o (V)'},
        {'y': bb_dcm['output_voltage'], 'label': 'v_o DCM', 'ylabel': 'v_o (V)'},
    ],
    xlabel='Time (us)',
    title='Buck-Boost Converter Waveforms'
)
plt.show()